In [ ]:
import pandas as pd 
import numpy as np 
from sklearn.feature_extraction.text import  TfidfVectorizer, CountVectorizer
from sklearn.pipeline import Pipeline



In [ ]:
x_train=pd.read_csv(r"D:\ci_dvc\ml_pieline\data\processed\train_processed.csv")

In [3]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

mlflow.set_experiment("vectorizer_comparison_logreg")

def eval_and_log(pipe, X_val, y_val):
    y_pred = pipe.predict(X_val)

    acc = accuracy_score(y_val, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_val, y_pred, average="binary", zero_division=0)

    mlflow.log_metric("val_accuracy", float(acc))
    mlflow.log_metric("val_precision", float(prec))
    mlflow.log_metric("val_recall", float(rec))
    mlflow.log_metric("val_f1", float(f1))

    # AUC (if available)
    try:
        if hasattr(pipe, "predict_proba"):
            y_proba = pipe.predict_proba(X_val)[:, 1]
            mlflow.log_metric("val_roc_auc", float(roc_auc_score(y_val, y_proba)))
        elif hasattr(pipe, "decision_function"):
            y_score = pipe.decision_function(X_val)
            mlflow.log_metric("val_roc_auc", float(roc_auc_score(y_val, y_score)))
    except Exception:
        pass

    return f1

with mlflow.start_run(run_name="parent_vectorizer_test") as parent:

    configs = [
        ("tfidf", TfidfVectorizer(ngram_range=(1,2), max_features=50000)),
        ("count", CountVectorizer(ngram_range=(1,2), max_features=50000)),
    ]

    best = {"name": None, "f1": -np.inf, "run_id": None}

    for vec_name, vec in configs:
        with mlflow.start_run(run_name=f"logreg_{vec_name}", nested=True) as child:

            pipe = Pipeline([
                ("vec", vec),
                ("model", LogisticRegression(max_iter=5000, C=1.0))
            ])

            # log what you are testing
            mlflow.log_param("vectorizer", vec_name)
            mlflow.log_param("vec_ngram_range", str(vec.ngram_range))
            mlflow.log_param("vec_max_features", vec.max_features)
            mlflow.log_param("model", "LogisticRegression")
            mlflow.log_params(pipe.named_steps["model"].get_params())

            pipe.fit(X_train, y_train)
            f1 = eval_and_log(pipe, X_val, y_val)

            mlflow.sklearn.log_model(pipe, artifact_path="model")

            if f1 > best["f1"]:
                best.update({"name": vec_name, "f1": float(f1), "run_id": child.info.run_id})

    mlflow.log_param("best_vectorizer", best["name"])
    mlflow.log_metric("best_val_f1", best["f1"])
    mlflow.log_param("best_child_run_id", best["run_id"])

print("Best vectorizer:", best)

c:\Users\adity\anaconda3\Lib\site-packages\pydantic\_internal\_fields.py:161: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
2026/02/26 17:13:55 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/02/26 17:13:55 INFO mlflow.store.db.utils: Updating database tables
2026/02/26 17:13:55 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/26 17:13:55 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/02/26 17:13:55 INFO alembic.runtime.migration: Running upgrade  -> 451aebb31d03, add metric step
2026/02/26 17:13:55 INFO alembic.runtime.migration: Running upgrade 451aebb31d03 -> 90e64c465722, migrate user column to tags
2026/02/26 17:13:55 INFO alembic.runtime.migration: Running upgrade 90e64c465722 -> 181f10493468, allow nulls for metric values
2026/02/26 17:13:55 INFO alembic.runtime.mig

NameError: name 'X_train' is not defined